In [0]:
from pyspark.sql.functions import *

recommendation_df = spark.table("workspace.myra_invest.gold_stock_recommendation_engine")

print(f"Recomendation Records: {recommendation_df.count()}")
display(recommendation_df.limit(10))

recommendation_df = recommendation_df.withColumn(
    "trendScore",
    when(
        (col("close") > col("movingAvg5")) &
        (col("close") > col("movingAvg20")),
        30
    )
    .when(col("close") > col("movingAvg5"), 20)
    .when(col("close") > col("movingAvg20"), 15)
    .otherwise(0)
)

display(
    recommendation_df.select(
        "companyName",
        "close",
        "movingAvg5",
        "movingAvg20",
        "trendScore"
    )
)

In [0]:
recommendation_df = recommendation_df.withColumn(
    "momentumGapPct",
    round(
        ((col("movingAvg5") - col("movingAvg20")) / col("movingAvg20")) * 100,
        2
    )
)

recommendation_df = recommendation_df.withColumn(
    "momentumScore",
    when(col("momentumGapPct") > 2, 30)
    .when(col("momentumGapPct") > 0, 25)
    .when(col("momentumGapPct") == 0, 15)
    .otherwise(5)
)

display(
    recommendation_df.select(
        "companyName",
        "movingAvg5",
        "movingAvg20",
        "momentumGapPct",
        "momentumScore"
    )
)

In [0]:
recommendation_df = recommendation_df.withColumn(
    "returnScore",
    when(col("dailyReturnPct") > 2, 20)
    .when(col("dailyReturnPct") > 1, 15)
    .when(col("dailyReturnPct") >= 0, 10)
    .otherwise(0)
)

In [0]:
recommendation_df = recommendation_df.withColumn(
    "priceChangeScore",
    when(col("priceChangePct") > 3, 20)
    .when(col("priceChangePct") > 2, 15)
    .when(col("priceChangePct") > 1, 10)
    .otherwise(5)
)

In [0]:
(
    recommendation_df.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable("workspace.myra_invest.gold_stock_recommendation_engine")
)

print("✅ Recommendation Engine table updated with Technical Score.")